In [ ]:
# ruff: noqa

import tidalseis.data.network_catalog as network_catalog
from functools import partial

import pandas as pd
import obspy

from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from tidalseis.datetime_ops import is_time_between
from tidalseis.load.network import (
    read_network_traces,
    link_station_trace_paths,
    flatten_station_traces,
    get_network_stream,
)
from tidalseis.catalog.triggering import (
    run_coincidece_trigger,
    TriggeringConfig,
)
from tidalseis.catalog.preprocess import PreprocessingConfig

from tidalseis.catalog.bundle_streams import bundle_network
from tidalseis.catalog.create import create_coincidence_catalog, get_stream_filepaths

import matplotlib.pyplot as plt
import cmap
import matplotlib.colors as mcolor

from tidalseis.catalog.models import CatalogModel, EventData

plt.switch_backend("qtagg")

NETWORK = network_catalog.AMERY_ICE_SHELF
BASE = Path("D:/seismic_data/amery_ice_shelf/trace_data/")

In [ ]:
preprocessing_config = PreprocessingConfig(
    detrend=True,
    filter_type="Bandpass",
    low_frequency_cutoff=5,
    high_frequency_cutoff=20
)

bundle_network(
    "D:/seismic_data/amery_ice_shelf/trace_data/",
    NETWORK["network_start"],
    NETWORK["network_end"],
    "D:/seismic_data/amery_ice_shelf/stream_data/",
    preprocessing_config
)

In [ ]:
triggering_config = TriggeringConfig(
    trigger_type="classicstalta",
    trigger_on_threshold=6,
    trigger_off_threshold=5,
    num_coincident_stations=3,
    long_term_average_length=60,
    short_term_average_length=2
)
stream_dir = Path("D:/seismic_data/amery_ice_shelf/stream_data/")
streams = get_stream_filepaths(stream_dir)

NDAYS = 15
all_events = create_coincidence_catalog(streams[:NDAYS])

In [ ]:
event_list: list[EventData] = []

for trigger in all_events:
    event = EventData(
        event_start=trigger["time"].datetime,
        event_duration=trigger["duration"],
        stations=trigger["stations"],
    )
    event_list.append(event)

catalog = CatalogModel(events=event_list)
with open("D:/seismic_data/amery_ice_shelf/event_catalog.json", "w") as f:
    f.write(catalog.model_dump_json(indent=2))

In [4]:
with open("D:/seismic_data/amery_ice_shelf/event_catalog.json", "r") as f:
    catalog = CatalogModel.model_validate_json(f.read())

In [17]:
times = catalog.get_relative_times(catalog.events[0].event_start)
nbins = (catalog.events[-1].event_start - catalog.events[0].event_start).total_seconds() // 3600
_ = plt.hist(times, bins = int(nbins))
plt.show()

In [ ]:
tides = pd.read_csv("D:/seismic_data/amery_ice_shelf/tidal_model.txt", delimiter=" ")
tide_height = tides.z
tide_date = tides.date
tide_time = tides.time
tide_datetime: list[tuple[datetime, float]] = []
FMT = "%m-%d-%Y%H:%M:%S"
for i, j, z in zip(tide_date, tide_time, tide_height):
    tide_datetime.append((datetime.strptime(i+j, FMT), float(z)))

In [ ]:
base_time = all_triggers[0]["time"]
last_time = all_triggers[-1]["time"]
nbins = float(last_time-base_time) // (3600)
event_timing: list[float] = [float(i["time"] - base_time) for i in all_triggers]
clipped_datetime = np.array([(i-base_time.datetime).total_seconds() for i, j in tide_datetime if i>base_time.datetime and i<last_time.datetime])
clipped_tide_height = np.array([j for i, j in tide_datetime if i>base_time.datetime and i<last_time.datetime])

tide_rate = np.gradient(clipped_tide_height, 3600)

# hist = np.histogram(event_timing, bins=nbins)
f, ax = plt.subplots()
ax2 = ax.twinx()
ax3 = ax.twinx()
_= ax.hist(event_timing, int(nbins), color='k')
ax2.plot(clipped_datetime, clipped_tide_height, color='r', label="Tide Height")
ax3.plot(clipped_datetime, -tide_rate, color='b', label="Tide Rate")
ax2.set_yticks([])
ax3.set_yticks([])
ax.set_ylim(0, 20)
ax2.legend(loc="upper right")
ax3.legend(loc="lower right")

# f, ax = plt.subplots()
# ax2 = ax.twinx()
# ax3 = ax.twinx()
# _= ax.hist(event_timing, int(nbins), color='k')
# ax2.plot(clipped_datetime, tide_rate, color='r')
# ax.set_ylim(0, 20)
# ax2.set_yticks([])
# ax3.set_yticks([])
# # plt.xticks(ticks=np.arange(0, 360), labels=clipped_datetime)
plt.show()

In [ ]:
event_list: list[tuple[datetime, list[np.ndarray]]] = []
clr_map = cmap.Colormap("crameri:hawaii").to_mpl()

subset_triggers = []
rng = np.random.default_rng()
ntot = min(10, len(triggers))
for i in rng.choice(np.arange(0, len(triggers)), ntot, replace=False):
    subset_triggers.append((i, triggers[i]))

for event, trig_dict in subset_triggers:
    f, ax = plt.subplots()

    event_data: list[np.ndarray] = []
    norm = mcolor.Normalize(0, len(trig_dict["trace_ids"]))
    for n, id in enumerate(trig_dict["trace_ids"]):
        trace = trace_dict[id]
        trig_start = trig_dict["time"]
        trig_end = trig_dict["time"] + trig_dict["duration"]
        trig_trace = trace.slice(starttime=trig_start-1, endtime=trig_end)
        trig_trace.normalize()
        ax.plot(trig_trace.times(), trig_trace.data + n, color=clr_map(norm(n)))
        ax.axvline(1, color='r', linestyle='--')
        event_data.append(np.stack((trig_trace.times(),trig_trace.data), axis=-1))
    
    event_time: datetime = trig_dict["time"].datetime
    event_list.append((event_time, event_data))
    ax.set_title(f"Event: {event+1} @ {event_time.strftime("%B %d, %Y // %I:%M:%S%p")}")
    event+=1

# Add stations to metadata
print(len(event_list))
plt.show()